In [28]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

In [29]:
in_dir = "../../results/02_tfbs_prediction/04_train"
out_dir = "../../results/02_tfbs_prediction/05_eval"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
for cell_type in ["K562", "HepG2"]:
    os.makedirs(f"{out_dir}/{cell_type}", exist_ok=True)

    # list all prediction files for the current cell type
    all_files = glob.glob(f"{in_dir}/{cell_type}/seq/prediction/*.csv")
    all_files = [os.path.basename(f) for f in all_files if f.endswith(".csv")]

    # find all tfs
    tfs = [f.replace(".csv", "") for f in all_files]  # replace with actual code to find all transcription factors for the given cell type

    df_precision_all_tfs = []
    df_aupr_all_tfs = []
    for tf in tfs:

        df_curve_list = []
        df_precision = []
        df_aupr = []
        for method in ["seq", "accessatac_tn5", "accessatac_dddss", "accessatac_both"]:
            df = pd.read_csv(f"{in_dir}/{cell_type}/{method}/prediction/{tf}.csv")
            precision, recall, thresholds = precision_recall_curve(df["true"], 
                                                                   df["pred"], 
                                                                   drop_intermediate=True)

            df_curve_list.append(pd.DataFrame({"precision": 
            precision, "recall": recall, 
            "method": method, "tf": tf}))

            # compute precision at certain target recall levels
            for target_recall in [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]:
                mask = recall >= target_recall
                if mask.sum() == 0:
                    _precision = 0.0
                else:
                    _precision = precision[mask].max() if mask.sum() > 0 else 0.0
                df_precision.append(pd.DataFrame({"target_recall": [target_recall], 
                                                  "precision": [_precision], 
                                                  "method": [method]}))
            # compute area under the precision-recall curve
            aupr = auc(recall, precision)
            df_aupr.append(pd.DataFrame({"aupr": [aupr], "method": [method]}))

        # plot 
        df_curve = pd.concat(df_curve_list, 
                             ignore_index=True)
        fig, ax = plt.subplots(figsize=(8, 6))

        sns.lineplot(data=df_curve, x="recall", 
                     y="precision", 
                     hue="method", ax=ax)

        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"Precision-Recall Curve for {tf} in {cell_type}")
        ax.legend()

        # save results
        df_curve.to_csv(f"{out_dir}/{cell_type}/{tf}_precision_recall.csv", 
                        index=False)

        fig.savefig(f"{out_dir}/{cell_type}/{tf}_precision_recall_curve.png")

        plt.close(fig)

        # combine precision at recall dataframes
        df_precision = pd.concat(df_precision, ignore_index=True)
        df_precision["tf"] = tf
        df_precision_all_tfs.append(df_precision)
        df_aupr_all_tfs.append(pd.concat(df_aupr, ignore_index=True))

    df_precision_all_tfs = pd.concat(df_precision_all_tfs, ignore_index=True)
    df_precision_all_tfs.to_csv(f"{out_dir}/{cell_type}/precision_at_recall.csv", index=False)

    df_aupr_all_tfs = pd.concat(df_aupr_all_tfs, ignore_index=True)
    df_aupr_all_tfs.to_csv(f"{out_dir}/{cell_type}/aupr.csv", index=False)

    # plot boxplot for AUPR
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(data=df_aupr_all_tfs, 
                x="method", 
                y="aupr", 
                ax=ax)
    ax.set_xlabel("Method")
    ax.set_ylabel("AUPR")
    ax.set_title(f"AUPR for {cell_type}")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/{cell_type}/aupr_boxplot.png")
    plt.close(fig)

    # plot boxplot 
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(data=df_precision_all_tfs, 
                x="target_recall", 
                y="precision", hue="method", 
                ax=ax)
    ax.set_xlabel("Target Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"Precision at Target Recall for {cell_type}")
    ax.legend(title="Method")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/{cell_type}/precision_at_recall_boxplot.png")
    plt.close(fig)